# Weather Library Tutorial

## Step-by-Step Guide to Using the LCCC Weather Library

This notebook provides a complete walkthrough of the Weather library for renewable energy forecasting using Monte Carlo sampling of historical weather patterns.

**What You'll Learn:**
- Load historical weather data using the LocalDataLoader
- Create a WeatherData sampler with HistoricalMetadata
- Generate Monte Carlo weather samples for future periods
- Extract and analyze forecast results
- Visualize time series and ensemble forecasts

## Prerequisites

### Installation
Install the weather library and dependencies:
```bash
# Option 1: Using uv (recommended)
pip install uv
uv sync --group notebook

# Option 2: Install from repo in development mode
pip install -e .
pip install jupyter ipykernel matplotlib seaborn plotly
```

### Data Requirements
This tutorial requires:
1. Historical calibrated wind weather data (NPY format)
2. Manifest file describing the data (JSON)
3. Prefix histograms (optional, for optimization)

Data should be placed in the configured `DOWNLOAD_DATA_DIR/calibrated/` directory.

## Step 1: Import Libraries

In [ ]:
import datetime
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Weather library imports
from weather.core.data_loader import LocalDataLoader
from weather.simulation.weather_data import HistoricalMetadata, WeatherData

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")

## Step 2: Initialize the Data Loader

The `LocalDataLoader` is the entry point for accessing local weather data, plant configuration, and generation data.

In [ ]:
# Initialize data loader with default path (configurable via DOWNLOAD_DATA_DIR env var)
loader = LocalDataLoader()

print("✓ Data loader initialized")
print(f"  Type: {type(loader).__name__}")
print(f"  Ready to load: weather data, plant data, generation data")

## Step 3: Load Historical Weather Manifest

Check what historical weather data is available. The manifest validates the NPY file and provides metadata.

In [ ]:
# Load and validate the historical weather manifest
manifest_wind = loader.check_historical_weather()

print("✓ Historical weather manifest loaded")
print(f"\nManifest Information:")
print(f"  Basename: {manifest_wind['basename']}")
print(f"  Horizon (start): {manifest_wind['horizon_utc']['start']}")
print(f"  Horizon (end): {manifest_wind['horizon_utc']['end']}")
print(f"  Columns: {manifest_wind['artifact']['columns']}")
print(f"  Size (bytes): {manifest_wind['artifact']['size_bytes']}")
print(f"  Rows per block (histogram): {manifest_wind['artifact_histogram']['rows_per_block']}")
print(f"  SHA256: {manifest_wind['artifact']['sha256'][:16]}...")

## Step 4: Create HistoricalMetadata Object

Convert the manifest into a `HistoricalMetadata` descriptor that the sampler uses.

In [ ]:
# Create metadata from manifest
metadata_wind = HistoricalMetadata(
    npy_basename=manifest_wind["basename"],
    path_resolver=loader.path_resolver_weather_data,
    data_limit_left=datetime.datetime.fromisoformat(manifest_wind["horizon_utc"]["start"]),
    data_limit_right=datetime.datetime.fromisoformat(manifest_wind["horizon_utc"]["end"]),
    columns=manifest_wind["artifact"]["columns"],
    hours_per_block=manifest_wind["artifact_histogram"]["rows_per_block"],
)

print("✓ HistoricalMetadata object created")
print(f"\nMetadata Details:")
print(f"  Data Coverage: {metadata_wind.data_limit_left} to {metadata_wind.data_limit_right}")
print(f"  Total Years: {(metadata_wind.data_limit_right - metadata_wind.data_limit_left).days / 365.25:.1f}")
print(f"  Columns: {metadata_wind.columns}")
print(f"  Column Count: {len(metadata_wind.columns)}")

## Step 5: Load Supporting Data (Histograms and Historical)

Load prefix histograms and historical weather data needed for advanced sampling features.

In [ ]:
# Load supporting data
prefix_histograms_wind = loader.get_prefix_histograms()
historical_data_wind = loader.get_historical_weather()

print("✓ Supporting data loaded")
if prefix_histograms_wind is not None:
    print(f"  Prefix histograms shape: {prefix_histograms_wind.shape}")
    print(f"  Type: {type(prefix_histograms_wind)}")
else:
    print(f"  Prefix histograms: Not available")

if historical_data_wind is not None:
    print(f"  Historical data shape: {historical_data_wind.shape}")
    print(f"  Type: {type(historical_data_wind)}")
else:
    print(f"  Historical data: Not available")

## Step 6: Initialize Weather Sampler

Create a `WeatherData` object configured for sampling. Optional: specify desired averages to constrain the output.

In [ ]:
# Initialize the weather sampler
# desired_averages: {column_index: [target_avg_1, target_avg_2, ...]}
# This constrains the sampler to match specified average values using inverse-CDF resampling

wind_sampler = WeatherData(
    metadata=metadata_wind,
    desired_averages={1: [0.14, 0.5, 0.66]},  # Three target averages for column 1
    prefix_histograms=prefix_histograms_wind,
    historical_data=historical_data_wind,
    ignore_zeros=False,  # Set True for solar (preserve zero nighttime values)
)

print("✓ Weather sampler initialized")
print(f"  Sampler type: {type(wind_sampler).__name__}")
print(f"  Desired averages: Column 1 -> [0.14, 0.5, 0.66]")
print(f"  Draw period: {wind_sampler.draw_period} days")

## Step 7: Generate a Single Weather Sample

Create one Monte Carlo sample for a future forecast period.

In [ ]:
# Define the forecast period
start_date = datetime.datetime(2025, 4, 1)
end_date = datetime.datetime(2025, 8, 7)

# Generate a single sample with fixed random seeds for reproducibility
sample_1 = wind_sampler.random_sample(
    future_start_date=start_date,
    future_end_date=end_date,
    python_rng=random.Random(42),           # Python random seed
    numpy_rng=np.random.default_rng(42),    # NumPy random seed
)

print("✓ Single weather sample generated")
print(f"\nSample Details:")
print(f"  Forecast Period: {start_date.date()} to {end_date.date()}")
print(f"  Duration: {(end_date - start_date).days} days")
print(f"  Sample shape: {sample_1.shape}")
print(f"  Data type: {sample_1.dtype}")

## Step 8: Extract Data from Sample

Access specific columns from the generated sample and view the data.

In [ ]:
# Extract all data from the sample (hourly time series)
# The sample is a 2D array where each row is an hour
# Each column is a different variable (as specified in metadata.columns)

# For simplicity, use the first column of the sample
wind_speed_data = sample_1[:, 0]

print("✓ Data extracted from sample")
print(f"\nWind Speed Data (Column 0):")
print(f"  Length: {len(wind_speed_data)} hours")
print(f"  Duration: {len(wind_speed_data) / 24:.1f} days")

# Display first 10 values
print(f"\n  First 10 hourly values:")
for i, val in enumerate(wind_speed_data[:10]):
    print(f"    Hour {i}: {val:.4f}")

# Basic statistics
print(f"\n  Statistics:")
print(f"    Min: {np.min(wind_speed_data):.4f}")
print(f"    Max: {np.max(wind_speed_data):.4f}")
print(f"    Mean: {np.mean(wind_speed_data):.4f}")
print(f"    Std Dev: {np.std(wind_speed_data):.4f}")
print(f"    Median: {np.median(wind_speed_data):.4f}")

## Step 9: Visualize Time Series

Plot the complete forecast time series.

In [ ]:
# Create hourly time index
time_index = pd.date_range(start=start_date, periods=len(wind_speed_data), freq='h')

# Create figure
fig, ax = plt.subplots(figsize=(15, 6))

# Plot time series
ax.plot(time_index, wind_speed_data, linewidth=1.5, color='steelblue', label='Wind Speed')
ax.fill_between(time_index, wind_speed_data, alpha=0.2, color='steelblue')

# Add mean line
mean_val = np.mean(wind_speed_data)
ax.axhline(y=mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}', alpha=0.7)

# Formatting
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Wind Speed', fontsize=12, fontweight='bold')
ax.set_title('Weather Forecast: Time Series (Single Monte Carlo Sample)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"✓ Time series visualization complete")

## Step 10: Statistical Summary

Compute detailed statistics of the forecast sample.

In [ ]:
# Calculate comprehensive statistics
stats_dict = {
    'Metric': ['Count', 'Mean', 'Std Dev', 'Min', '25%', 'Median', '75%', 'Max', 'Range'],
    'Value': [
        len(wind_speed_data),
        f"{np.mean(wind_speed_data):.6f}",
        f"{np.std(wind_speed_data):.6f}",
        f"{np.min(wind_speed_data):.6f}",
        f"{np.percentile(wind_speed_data, 25):.6f}",
        f"{np.median(wind_speed_data):.6f}",
        f"{np.percentile(wind_speed_data, 75):.6f}",
        f"{np.max(wind_speed_data):.6f}",
        f"{np.max(wind_speed_data) - np.min(wind_speed_data):.6f}",
    ]
}

stats_df = pd.DataFrame(stats_dict)
print("\n✓ Statistical Summary:")
print(stats_df.to_string(index=False))

## Step 11: Distribution Analysis

Visualize the distribution through histograms and box plots.

In [ ]:
# Create subplots for distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(wind_speed_data, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
mean_val = np.mean(wind_speed_data)
axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
axes[0].set_xlabel('Wind Speed', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency (hours)', fontsize=11, fontweight='bold')
axes[0].set_title('Distribution: Histogram', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot
bp = axes[1].boxplot(wind_speed_data, vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('steelblue')
axes[1].set_ylabel('Wind Speed', fontsize=11, fontweight='bold')
axes[1].set_title('Distribution: Box Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"✓ Distribution analysis complete")

## Step 12: Generate Multiple Monte Carlo Samples (Ensemble)

Create an ensemble of weather samples to represent forecast uncertainty.

In [ ]:
# Generate multiple samples with different random seeds
num_samples = 10
ensemble_samples = []

print(f"Generating {num_samples} ensemble members...")

for i in range(num_samples):
    sample = wind_sampler.random_sample(
        future_start_date=start_date,
        future_end_date=end_date,
        python_rng=random.Random(1000 + i),        # Different seed for each sample
        numpy_rng=np.random.default_rng(1000 + i),
    )
    ensemble_samples.append(sample)
    print(f"  ✓ Sample {i+1}/{num_samples} generated")

print(f"\n✓ Ensemble of {num_samples} samples generated")

# Extract data from all samples
ensemble_data = np.array([sample[:, 0] for sample in ensemble_samples])  # Shape: (num_samples, hours)
print(f"  Ensemble shape: {ensemble_data.shape}")

## Step 13: Ensemble Visualization with Uncertainty Bands

Plot all ensemble members with mean and percentile confidence bounds.

In [ ]:
# Calculate ensemble statistics
ensemble_mean = np.mean(ensemble_data, axis=0)
ensemble_p10 = np.percentile(ensemble_data, 10, axis=0)   # 10th percentile
ensemble_p90 = np.percentile(ensemble_data, 90, axis=0)   # 90th percentile
ensemble_p25 = np.percentile(ensemble_data, 25, axis=0)   # 25th percentile
ensemble_p75 = np.percentile(ensemble_data, 75, axis=0)   # 75th percentile
ensemble_median = np.percentile(ensemble_data, 50, axis=0)

# Create visualization
fig, ax = plt.subplots(figsize=(15, 7))

# Plot individual ensemble members (light gray)
for i, data in enumerate(ensemble_data):
    if i == 0:
        ax.plot(time_index, data, linewidth=0.7, alpha=0.25, color='gray', label='Individual samples')
    else:
        ax.plot(time_index, data, linewidth=0.7, alpha=0.25, color='gray')

# Plot percentile bands
ax.fill_between(time_index, ensemble_p10, ensemble_p90, alpha=0.3, color='orange', label='P10-P90 (80% confidence)')
ax.fill_between(time_index, ensemble_p25, ensemble_p75, alpha=0.4, color='coral', label='P25-P75 (50% confidence)')

# Plot mean and median
ax.plot(time_index, ensemble_mean, linewidth=2.5, color='darkred', label='Ensemble Mean', zorder=10)
ax.plot(time_index, ensemble_median, linewidth=2, color='darkblue', label='Ensemble Median', linestyle='--', zorder=9)

# Formatting
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Wind Speed', fontsize=12, fontweight='bold')
ax.set_title(f'Ensemble Forecast with Uncertainty Bands ({num_samples} Monte Carlo Samples)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"✓ Ensemble visualization complete")

## Step 14: Ensemble Uncertainty Analysis

Analyze the ensemble spread and variability.

In [ ]:
# Calculate ensemble-level statistics
ensemble_uncertainty = np.std(ensemble_data, axis=0)  # Standard deviation across samples

ensemble_stats = {
    'Metric': [
        'Number of Samples',
        'Overall Mean',
        'Overall Std Dev',
        'Global Min',
        'Global Max',
        'P10 (10th percentile)',
        'P50 (median)',
        'P90 (90th percentile)',
        'Avg Uncertainty (Std Dev)',
        'Max Uncertainty',
        'Min Uncertainty'
    ],
    'Value': [
        num_samples,
        f"{np.mean(ensemble_mean):.4f}",
        f"{np.std(ensemble_mean):.4f}",
        f"{np.min(ensemble_data):.4f}",
        f"{np.max(ensemble_data):.4f}",
        f"{np.percentile(ensemble_data, 10):.4f}",
        f"{np.percentile(ensemble_data, 50):.4f}",
        f"{np.percentile(ensemble_data, 90):.4f}",
        f"{np.mean(ensemble_uncertainty):.4f}",
        f"{np.max(ensemble_uncertainty):.4f}",
        f"{np.min(ensemble_uncertainty):.4f}",
    ]
}

ensemble_stats_df = pd.DataFrame(ensemble_stats)
print("\n✓ Ensemble Statistical Summary:")
print(ensemble_stats_df.to_string(index=False))

## Step 15: Uncertainty Over Time

Visualize how forecast uncertainty varies throughout the period.

In [ ]:
# Create subplots showing uncertainty
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Top: Ensemble mean with uncertainty band
axes[0].plot(time_index, ensemble_mean, linewidth=2, color='darkred', label='Mean', zorder=5)
axes[0].fill_between(time_index, 
                      ensemble_mean - ensemble_uncertainty, 
                      ensemble_mean + ensemble_uncertainty, 
                      alpha=0.3, color='red', label='± 1 Std Dev')
axes[0].set_ylabel('Wind Speed', fontsize=11, fontweight='bold')
axes[0].set_title('Ensemble Mean with Uncertainty Envelope', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Bottom: Uncertainty (std dev) over time
axes[1].fill_between(time_index, ensemble_uncertainty, alpha=0.4, color='orange', label='Forecast Uncertainty')
axes[1].plot(time_index, ensemble_uncertainty, linewidth=1.5, color='darkorange')
axes[1].axhline(y=np.mean(ensemble_uncertainty), color='red', linestyle='--', linewidth=2, 
                 label=f'Average Uncertainty: {np.mean(ensemble_uncertainty):.4f}')
axes[1].set_xlabel('Date', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Std Dev (Uncertainty)', fontsize=11, fontweight='bold')
axes[1].set_title('Forecast Uncertainty Over Time', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"✓ Uncertainty analysis visualization complete")

## Key Takeaways

### Library Components
- **LocalDataLoader**: Access weather data, plant data, and generation records
- **HistoricalMetadata**: Descriptor for historical weather data with validation
- **WeatherData**: Monte Carlo sampler for generating future weather scenarios

### Workflow
1. Initialize `LocalDataLoader`
2. Load and validate historical weather manifest
3. Create `HistoricalMetadata` object
4. Initialize `WeatherData` sampler with optional constraints
5. Generate samples for future periods
6. Extract and analyze results

### Monte Carlo Sampling
- Multiple samples represent forecast uncertainty
- Percentile bands show confidence intervals
- Ensemble mean provides best estimate
- Random seeds ensure reproducibility

### Customization
- **Desired averages**: Constrain forecast to match target values using inverse-CDF resampling
- **Random seeds**: Control reproducibility across runs
- **Ensemble size**: Balance accuracy vs. computation time
- **ignore_zeros**: For solar data, preserves nighttime zeros

### Next Steps
- Convert wind speed to power output using plant models
- Analyze financial implications of uncertainty
- Integrate with grid planning workflows
- Implement custom calibration pipelines